In [1]:
import numpy as np
import pandas as pd
import time, warnings

In [24]:
data_path = "data/crypto_market_data_daily.csv"
output_csv = "data/bubble_labels.csv"

In [25]:
rolling_k = 30
adf_lags = 1
T_sim = 100
n_sim = 2000
seeds = 42
coins = ["BTC", "ETH", "BNB", "ADA", "SOL", "DOGE"]

In [26]:
def adf_t_stat(y, lags = 1):
    y = np.asarray(y, dtype = np.float64)
    diff_y = np.diff(y)
    n = len(diff_y) - lags
    if n < lags + 3:
        return np.nan
    X = np.empty((n, 2 + lags))
    X[:, 0] = 1.0
    X[:, 1] = y[lags : lags + n]
    for j in range(1, lags + 1):
        X[:, 1 + j] = diff_y[lags - j: lags - j + n]
    Y = diff_y[lags: lags + n]
    try: 
        XtXi = np.linalg.inv(X.T @ X)
    except np.linalg.LinAlgError:
        return np.nan
    b = XtXi @ (X.T @ Y)
    resid = Y - X @ b
    s_2 = (resid @ resid) / max(n - X.shape[1], 1)
    s_e = np.sqrt(s_2 * XtXi[1, 1])
    return np.nan if s_e <= 0 else b[1] / s_e

In [27]:
def bsadf_seq(log_price, r0, lags = 1):
    T = len(log_price)
    bsadf = np.full(T, np.nan)
    for t2 in range(r0, T):
        best = -np.inf
        for r1 in range(0, t2 - r0 + 1):
            s = adf_t_stat(log_price[r1 : t2 + 1], lags = lags)
            if not np.isnan(s) and s > best:
                best = s
        if best > -np.inf:
            bsadf[t2] = best
    return bsadf

In [28]:
def r0_psy(T):
    return max(int(np.ceil((0.01 + 1.8 / np.sqrt(T)) * T)), adf_lags + 5)

def sim_cv_base(T_simulation = 100, n_simulation = 2000, lags = 1, seed = 42, quantiles = (0.90, 0.95)):
    np.random.seed(seed)
    r0 = r0_psy(T_simulation)
    simulations = np.full((n_simulation, T_simulation), np.nan)
    for i in range(n_simulation):
        rw = np.cumsum(np.random.randn(T_simulation)) + np.arange(T_simulation) / T_simulation
        simulations[i] = bsadf_seq(rw, r0, lags = lags)
    return {q: np.nanquantile(simulations, q, axis = 0) for q in quantiles}

def map_cv_to_T(cv_base, T_simulation, T_actual, quantiles = (0.90, 0.95)):
    T_actual_index = np.arange(T_actual)
    T_simulation_index = T_actual_index / T_actual * T_simulation
    T_simulation_index = np.clip(T_simulation_index, 0, T_simulation - 1)

    mapped = {}
    for q in quantiles: 
        mapped[q] = np.interp(T_simulation_index, np.arange(T_simulation, dtype = float), cv_base[q])
    return mapped

In [29]:
def bubbles_label(log_price, bsadf, cv, k = 30):
    T = len(log_price)
    labels = np.full(T, "Not Bubble", dtype = object)
    for t in range(T):
        if np.isnan(bsadf[t]) or np.isnan(cv[t]):
            continue
        if bsadf[t] > cv[t]:
            ref = max(0, t - k)
            price_change = log_price[t] - log_price[ref]
            labels[t] = "Bubble Creation" if price_change > 0 else "Bubble Collapse"
    return labels

In [30]:
def main():
    raw = pd.read_csv(data_path)
    raw["timestamp"] = pd.to_datetime(raw["timestamp"])
    cv_base = sim_cv_base(T_simulation = T_sim, n_simulation = n_sim, lags = adf_lags, seed = seeds, quantiles = (0.90, 0.95))
    all_results = []
    for coin in coins: 
        df_c = (raw[raw["symbol"] == coin].sort_values("timestamp").reset_index(drop = True).copy())
        if df_c.empty:
            continue
        lp = np.log(df_c["close"].values)
        T = len(lp)
        r0 = r0_psy(T)
        bs = bsadf_seq(lp, r0, lags = adf_lags)
        cv_mapped = map_cv_to_T(cv_base, T_sim, T, quantiles = (0.90, 0.95))
        cv90 = cv_mapped[0.90]
        cv95 = cv_mapped[0.95]

        df_c["gsadf_label_95"] = bubbles_label(lp, bs, cv95, k = rolling_k)
        df_c["gsadf_label_90"] = bubbles_label(lp, bs, cv90, k = rolling_k)

        all_results.append(df_c)

    final = pd.concat(all_results, ignore_index = True)
    out_cols = ["timestamp", "symbol", "open", "high", "low", "close", "volume", "gsadf_label_95", "gsadf_label_90"]
    final[out_cols].to_csv(output_csv, index = False)

if __name__ == "__main__":
    main()

/opt/homebrew/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1617: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
